# Surrogate Selection — L5B15

Picks the surrogate-model architecture used for the rest of the thesis. The comparison is run on **L5B15** (primary luggage variant, largest sample) and benchmarks five families: Linear Regression, GaussianNB (architecture-matched to Pega ADM), RandomForest, LightGBM, and CatBoost — all on the same train/test split and the same Pega ADM target.

Outcome of §7.6: **CatBoost** is the selected surrogate. The chosen architecture is then refit per offer variant in `05_surrogate_fit.ipynb`, which is where downstream notebooks (06–08) read their model artifacts from.

**Run `02_data_ingestion.ipynb` first.**

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
# Selection is L5B15-only. Per-variant fitting lives in 05_surrogate_fit.ipynb.
from pathlib import Path

VARIANT       = "L5B15"
PROCESSED_DIR = Path("../data/processed")
PROCESSED_FILE = PROCESSED_DIR / "luggage_email_outbound.parquet"
ARTIFACT_DIR  = Path("../data/artifacts") / VARIANT
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Variant : {VARIANT}")
print(f"Input   : {PROCESSED_FILE}")
print(f"Artifacts → {ARTIFACT_DIR}  (only the comparison CSV is written here from this notebook)")

In [ ]:
import sys

import numpy as np
import pandas as pd
from scipy.stats import spearmanr as _spearmanr
from sklearn.model_selection import KFold, train_test_split

sys.path.insert(0, "../src")
from my_project.features import VARIANT_FEATURES
from my_project.metrics import fidelity_suite
from my_project.surrogate import (
    NaiveBayesBaseline,
    build_feature_matrix,
    evaluate_surrogate,
    train_catboost,
)

print("Imports OK")

In [3]:
df = pd.read_parquet(PROCESSED_FILE)
# Filter to primary variant (pyName preserves original casing e.g. "BookingDotCom")
df = df[df["pyName"] == VARIANT].reset_index(drop=True)
print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")
print(f"Model versions: {df['modelVersion'].nunique()}")
print(df["modelVersion"].value_counts().to_string())

Rows: 42,160  |  Columns: 441
Model versions: 1717
modelVersion
5c2acbce-86df-50a0-b3ba-17a5f6a5270c    446
5ed27c75-0039-5c7f-9bda-d950ccf254df    416
b9116607-3620-526a-96f2-490303931dae    414
c77ad3f1-48a3-5d06-8dcd-376561a6ae52    410
6db6f699-1d07-5d4e-9b7e-54cc1ca6f99b    392
42e89e31-ee09-5bac-96a0-658988831fd7    386
a878af03-6183-53de-9357-1a9b294d3dba    384
3f17039f-88f1-5435-ba32-890ad55b698e    369
ac8ac396-4031-57cf-bf11-54bbf465325b    367
f9161152-526a-5d3b-a24a-21e643a500e3    366
ece43c87-1d4d-5bc3-b310-27e32ab67f25    364
0777b91e-2eda-5471-880a-7a648e983f0f    362
b335bafa-1b51-5662-b6b9-1a8462c839b5    359
267c0f90-0fb5-544e-8be8-9a77ff9cdbdf    350
1af6e524-d196-53e5-8485-c4e133001a00    350
6718038b-cd9f-559f-8ce3-f84f7a09ffbc    344
ca20c65f-6850-5f60-8e6a-2d086c4183a9    336
6e5566c8-fa43-5eb8-8dbc-bd8900fc9e74    319
998c98a1-04bf-5d88-a1df-f1b5c19a7491    317
e1f7c11a-4fbb-5109-95d9-1b75bfe6342a    316
810f32f6-3e75-583f-9af9-364d834fb025    314
16451dfc-db7

## 7. Surrogate Model

The surrogate approximates the black-box Pega ADM function $f$ using only the logged tuples $(X_i, \hat{p}_i)$. It serves as the prediction oracle for SHAP and LIME in notebook 05. Fidelity is measured on a held-out test set.

In [4]:
### 7.1  Feature matrix
cfg = VARIANT_FEATURES[VARIANT]
X, y, cat_cols, num_cols = build_feature_matrix(df, list(cfg.features), cfg.numeric)
print(f"Features: {X.shape[1]}  ({len(num_cols)} numeric, {len(cat_cols)} categorical)")
print(f"Target   : {y.describe().to_dict()}")
X.head(3)

Features: 25  (10 numeric, 15 categorical)
Target   : {'count': 42160.0, 'mean': 0.002138611822724781, 'std': 0.0012181893457304505, 'min': 0.00047329387845813204, '25%': 0.001805314743700683, '50%': 0.001938127317989158, '75%': 0.002189295530415226, 'max': 0.032568238213399506}


,param::Param.BundleName,CustBookedFlight.BookingData.CultureCode,CustBookedFlight.FlightData.AirlineCodeIATA,CustBookedFlight.BookingData.FlightInboundArrival,CustBookedFlight.FlightData.DepartureAirport,CustBookedFlight.BookingData.DurationOfJourney,CustBookedFlight.BookingData.FlightInboundDeparture,IH.Push.Outbound.Pending.pxLastOutcomeTime.DaysSince,CustBookedFlight.Language,IH.Email.Outbound.Pending.pxLastOutcomeTime.DaysSince,...,CustBookedFlight.FlightData.DestinationAirport,CustBookedFlight.DepartureAirport,CustBookedFlight.IsStaffStandBy,IH.Email.Inbound.Clicked.pyHistoricalOutcomeCount,IH.Email.Inbound.Clicked.pxLastOutcomeTime.DaysSince,IH.Mobile.Inbound.Impression.pxLastOutcomeTime.DaysSince,IH.Email.Outbound.Clicked.pyHistoricalOutcomeCount,CustBookedFlight.SeatNumber,param::Param.FlightCost,param::Param.Age
0,BookingConfirmation,fr-FR,TO,__MISSING__,ORY,0.0,__MISSING__,NaN,French,NaN,...,TLN,ORY,__MISSING__,NaN,NaN,NaN,NaN,__MISSING__,264.0,56
1,__MISSING__,fr-FR,TO,__MISSING__,ORY,0.0,__MISSING__,NaN,French,NaN,...,TLN,ORY,__MISSING__,NaN,NaN,NaN,NaN,__MISSING__,NaN,__MISSING__
2,__MISSING__,__MISSING__,__MISSING__,__MISSING__,__MISSING__,7.0,__MISSING__,NaN,Dutch,NaN,...,__MISSING__,EIN,__MISSING__,NaN,NaN,NaN,NaN,__MISSING__,NaN,__MISSING__


In [5]:
### 7.2  Train / test split  (random 80/20, fixed seed)
# Temporal splits for stability analysis are defined in notebook 06.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

Train: 33,728  |  Test: 8,432


In [6]:
### 7.3  CatBoost — depth grid search (5-fold CV on Spearman ρ)
# The NB generating process is strictly additive (no feature interactions),
# so lower depths should generalise better. CV selects the optimal depth.

kf = KFold(n_splits=5, shuffle=True, random_state=42)
depth_results = []

for depth in range(1, 7):
    fold_rhos = []
    for tr_idx, val_idx in kf.split(X_train):
        m = train_catboost(
            X_train.iloc[tr_idx], y_train.iloc[tr_idx], cat_cols, depth=depth
        )
        rho, _ = _spearmanr(y_train.iloc[val_idx], m.predict(X_train.iloc[val_idx]))
        fold_rhos.append(rho)
    depth_results.append({
        "depth":    depth,
        "mean_rho": round(float(np.mean(fold_rhos)), 4),
        "std_rho":  round(float(np.std(fold_rhos)),  4),
    })

cv_df = pd.DataFrame(depth_results).set_index("depth")
display(cv_df.style.format("{:.4f}").highlight_max(subset=["mean_rho"], color="#d4edda"))

best_depth = int(cv_df["mean_rho"].idxmax())
print(f"\nSelected depth: {best_depth}")

cb_model    = train_catboost(X_train, y_train, cat_cols, depth=best_depth)
cb_fidelity = evaluate_surrogate(cb_model, X_test, y_test, f"CatBoost (depth={best_depth})")

,mean_rho,std_rho
depth,,
1,0.3522,0.0090
2,0.6107,0.0089
3,0.6779,0.0073
4,0.7086,0.0077
5,0.7217,0.0065
6,0.7278,0.0095



Selected depth: 6


In [7]:
### 7.4  Naive Bayes robustness baseline
nb_model    = NaiveBayesBaseline(n_bins=10).fit(X_train, y_train, cat_cols, num_cols)
nb_fidelity = evaluate_surrogate(nb_model, X_test, y_test, "NaiveBayes")

In [8]:
### 7.5  Fidelity comparison
fidelity_df = (
    pd.DataFrame([cb_fidelity, nb_fidelity])
    .set_index("model")
    .rename(columns={
        "r2":           "R²",
        "rmse":         "RMSE",
        "spearman_rho": "Spearman ρ",
        "kendall_tau":  "Kendall τ",
        "ks_stat":      "KS",
    })
)
display(
    fidelity_df.style
    .format("{:.4f}")
    .highlight_max(subset=["R²", "Spearman ρ", "Kendall τ"], color="#d4edda")
    .highlight_min(subset=["RMSE", "KS"],                    color="#d4edda")
)

,R²,RMSE,Spearman ρ,Kendall τ,KS
model,,,,,
CatBoost (depth=6),0.9209,0.0004,0.7370,0.5401,0.2249
NaiveBayes,0.2199,0.0012,0.5387,0.3694,0.2949


### 7.6  Surrogate method comparison

Five surrogate families on the same train/test split and the same Pega ADM target:

- **Linear Regression** — lower-bound baseline (no interactions in the encoded space).
- **GaussianNB** — architecture-matched to Pega ADM's internal Naive Bayes. Target is binned into deciles and the classifier's probabilities are projected back to continuous predictions via class-probability-weighted bin means.
- **Random Forest** — tree-based, no boosting; isolates whether boosting specifically or general tree flexibility drives fidelity.
- **LightGBM** — boosting with native categorical handling, comparable to CatBoost.
- **CatBoost** — primary surrogate, depth-tuned in §7.3.

Linear / GaussianNB / Random Forest / LightGBM use `OrdinalEncoder(unknown_value=-1)` on cat columns with mean-imputed numerics. LightGBM gets the cat indices via `categorical_feature`. CatBoost reuses `X_train`/`X_test` directly with native cat handling.

In [ ]:
### 7.6.1  Shared encoded matrix for Linear / GaussianNB / RandomForest / LightGBM
# Cat columns first (so cat indices are 0..len(cat_cols)-1), numerics last with
# mean imputation. Unknown categories in the test set → -1 sentinel.
import lightgbm as lgb
from sklearn.ensemble    import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.naive_bayes  import GaussianNB
from sklearn.preprocessing import OrdinalEncoder

ord_enc      = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_tr_cat_enc = ord_enc.fit_transform(X_train[cat_cols].astype(str))
X_te_cat_enc = ord_enc.transform(X_test[cat_cols].astype(str))

num_means = X_train[num_cols].apply(pd.to_numeric, errors="coerce").mean()
X_tr_num  = (X_train[num_cols].apply(pd.to_numeric, errors="coerce")
                              .fillna(num_means).values)
X_te_num  = (X_test[num_cols].apply(pd.to_numeric, errors="coerce")
                             .fillna(num_means).values)

X_tr_enc    = np.hstack([X_tr_cat_enc, X_tr_num])
X_te_enc    = np.hstack([X_te_cat_enc, X_te_num])
cat_idx_enc = list(range(len(cat_cols)))  # cat cols occupy the first len(cat_cols) columns

print(f"Encoded matrix: train {X_tr_enc.shape}, test {X_te_enc.shape}")
print(f"Categorical indices (for LightGBM): {cat_idx_enc}")

In [ ]:
### 7.6.2  Fit five surrogates on the same train/test split and collect fidelity rows.
# CatBoost reuses cb_model from §7.3 (already random_seed=42, verbose=False).
# All sklearn-style models use the encoded matrix; LightGBM gets cat indices natively.

results = []

# Linear Regression — lower-bound, no interactions in the encoded space
lr_pred = LinearRegression().fit(X_tr_enc, y_train).predict(X_te_enc)
results.append({"Model": "LinearRegression", **fidelity_suite(y_test.values, lr_pred)})

# GaussianNB — classifier; bin target into deciles, recover via weighted bin means
y_tr_bins = pd.qcut(y_train, q=10, labels=False, duplicates="drop").astype(int)
bin_means = y_train.groupby(y_tr_bins).mean().sort_index().values
gnb       = GaussianNB().fit(X_tr_enc, y_tr_bins)
gnb_pred  = gnb.predict_proba(X_te_enc) @ bin_means
results.append({"Model": "GaussianNB", **fidelity_suite(y_test.values, gnb_pred)})

# Random Forest — tree flexibility, no boosting
rf_pred = RandomForestRegressor(
    n_estimators=100, random_state=42, n_jobs=-1,
).fit(X_tr_enc, y_train).predict(X_te_enc)
results.append({"Model": "RandomForest", **fidelity_suite(y_test.values, rf_pred)})

# LightGBM — boosting with native categorical handling on the ordinal-coded matrix
lgb_pred = lgb.LGBMRegressor(random_state=42, verbose=-1).fit(
    X_tr_enc, y_train,
    categorical_feature=cat_idx_enc,
).predict(X_te_enc)
results.append({"Model": "LightGBM", **fidelity_suite(y_test.values, lgb_pred)})

# CatBoost — reuse depth-tuned model from §7.3
results.append({"Model": "CatBoost", **fidelity_suite(y_test.values, cb_model.predict(X_test))})

comparison_df = (
    pd.DataFrame(results)[["Model", "r2", "rmse", "spearman_rho", "kendall_tau", "ks_stat"]]
    .rename(columns={
        "r2":           "R²",
        "rmse":         "RMSE",
        "spearman_rho": "Spearman ρ",
        "kendall_tau":  "Kendall τ",
        "ks_stat":      "KS statistic",
    })
)

display(
    comparison_df.style
    .format({"R²": "{:.4f}", "RMSE": "{:.6f}",
             "Spearman ρ": "{:.4f}", "Kendall τ": "{:.4f}", "KS statistic": "{:.4f}"})
    .highlight_max(subset=["R²", "Spearman ρ", "Kendall τ"], color="#d4edda")
    .highlight_min(subset=["RMSE", "KS statistic"],          color="#d4edda")
    .hide(axis="index")
)

In [ ]:
### 7.6.3  LaTeX table + persisted CSV
# pandas 3.x removed `booktabs` from DataFrame.to_latex(); use the Styler path,
# which emits \toprule/\midrule/\bottomrule when hrules=True.
latex_table = (
    comparison_df.style
    .format({"R²": "{:.4f}", "RMSE": "{:.6f}",
             "Spearman ρ": "{:.4f}", "Kendall τ": "{:.4f}", "KS statistic": "{:.4f}"})
    .hide(axis="index")
    .to_latex(hrules=True)
)
print(latex_table)

comparison_path = ARTIFACT_DIR / "surrogate_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
print(f"Saved → {comparison_path}")